# exp04: context k = 3 (3 segments before and after), separator "\n"

The adopted setting from exp03 (lr 1e-5, 3 epochs, fp16 + fused AdamW, batch 32), now with k = 3 instead of k = 0. The separator is `"\n"`, which DeBERTa's tokenizer turns into a plain space: the only structure the model sees is the `[T] … [/T]` markers around the target.
Seeds 42, 43, 44, to compare with exp03's k = 0 reference (0.5225 ± 0.0059).
Note: `tasks/C1-encoder-baseline/experiments/exp04_context-k3/NOTE.md`.

Inputs are ~4× longer than at k = 0 (81 tokens per batch on average, at most 270), so **section 5 first checks that batch 32 fits in memory**, with one real training step on the longest training batch.

Temporary trial (`scripts/trial_run.py --mode flat`); its numbers are **not reportable** (PROTOCOL §4.1).

Before running, in the notebook settings:
- **Accelerator:** GPU **T4 x2**: two independent runs at a time, one per GPU
- **Internet:** on (to clone the repo and download the model)
- **Input:** the private `smi-v4-meetings` dataset attached

The repo branch `C1/encoder-baseline` must be pushed: the notebook clones it from GitHub.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
Do **not** install `requirements.txt` here: it would replace Kaggle's own torch build.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C1/encoder-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 sentencepiece==0.2.2

## 3. Data folder
Found automatically: the folder that contains `source.json`, `train/`, `val/`, `test/`.

In [ ]:
import glob, os

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
print("source.json found at:", hits)
assert len(hits) == 1, "expected exactly one source.json under /kaggle/input"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "->", sorted(os.listdir(DATA_ROOT)))

## 4. Model loads in fp32
Also downloads the weights once (371 MB).

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/repo/src")
from smi.models.model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer("microsoft/deberta-v3-base")
print("dtype:", model.dtype)  # must be torch.float32
assert str(model.dtype) == "torch.float32"
del model

## 5. Settings and the memory check
One training step (fp16 + fused AdamW, as in `train()`) on the longest training batch of the epoch, in a separate process so the GPU is free again afterwards. Two steps, because AdamW creates its `m` and `v` on the first one.

**If the peak is close to the GPU's memory (above ~13.5 GB of ~15), stop here:** the run would need batch 16 with 2 accumulation steps.

In [ ]:
PAST_K = 3
FUTURE_K = 3
LR = 1e-5
EPOCHS = 3
SEEDS = [42, 43, 44]
K0_REFERENCE = {42: 0.5250, 43: 0.5267, 44: 0.5157}  # exp03 : lr 1e-5 , 3 epochs , k = 0

In [ ]:
%%writefile /kaggle/working/memory_check.py
"""one real training step on the longest training batch ; prints the peak GPU memory."""
import sys

import torch

sys.path.insert(0, "src")
sys.path.insert(0, "scripts")
from smi.inputs.loader import build_dataloader
from smi.models.model import build_optimizer, load_model_and_tokenizer
from trial_run import SETTINGS

data_root, past_k, future_k, lr = sys.argv[1], int(sys.argv[2]), int(sys.argv[3]), float(sys.argv[4])
device = torch.device("cuda")
model, tokenizer = load_model_and_tokenizer("microsoft/deberta-v3-base")
model.to(device)
loader = build_dataloader(
    f"{data_root}/train", tokenizer, past_k=past_k, future_k=future_k, batch_size=SETTINGS["batch_size"],
    chunk_batches=SETTINGS["chunk_batches"], shuffle=True, max_length=SETTINGS["max_length"],
    separator=SETTINGS["separator"], markers=tuple(SETTINGS["markers"]), seed=42,
)

# the batch with the most tokens (batch size × padded length)
longest = None
for batch in loader:
    if longest is None or batch["input_ids"].numel() > longest["input_ids"].numel():
        longest = batch
print("longest training batch (segments, tokens):", tuple(longest["input_ids"].shape))

optimizer = build_optimizer(model, lr, SETTINGS["weight_decay"], fused=True)
scaler = torch.amp.GradScaler("cuda")
loss_function = torch.nn.CrossEntropyLoss()
model.train()
torch.cuda.reset_peak_memory_stats()
for step in range(2):
    input_ids = longest["input_ids"].to(device)
    attention_mask = longest["attention_mask"].to(device)
    labels = longest["labels"].to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    loss = loss_function(logits.float(), labels)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), SETTINGS["max_grad_norm"])
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()
torch.cuda.synchronize()

total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"peak allocated {torch.cuda.max_memory_allocated() / 1e9:.1f} GB , peak reserved {torch.cuda.max_memory_reserved() / 1e9:.1f} GB , GPU total {total:.1f} GB")

In [ ]:
!cd /kaggle/working/repo && CUDA_VISIBLE_DEVICES=0 python /kaggle/working/memory_check.py {DATA_ROOT} {PAST_K} {FUTURE_K} {LR}

## 6. Run: a queue over both T4s
Each GPU takes the next run as soon as its current one ends: seeds 42 and 43 in parallel, then seed 44. Each process sees only one GPU (`CUDA_VISIBLE_DEVICES`), so there is no communication between them. Progress: the newest line of each running log every 2 minutes.

**Safe to rerun after a restart:**
- a run whose folder has a `metrics.json` is finished and is skipped;
- a folder without one is an interrupted run: it is deleted and the run starts again.

In [ ]:
import shutil, subprocess, time

RUNS_DIR = "/kaggle/working/runs"
LOGS_DIR = "/kaggle/working/logs"
os.makedirs(LOGS_DIR, exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
RUN_NAME = f"exp04_k{PAST_K}"


def start(gpu, seed):
    """one training process that sees only one GPU (CUDA_VISIBLE_DEVICES) ; its output goes to its own log file."""
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src")
    log_path = f"{LOGS_DIR}/{RUN_NAME}_seed{seed}_gpu{gpu}.log"
    cmd = ["python", "scripts/trial_run.py", "--mode", "flat", "--seeds", str(seed),
           "--lr", str(LR), "--epochs", str(EPOCHS), "--past-k", str(PAST_K), "--future-k", str(FUTURE_K),
           "--name", RUN_NAME, "--data-root", DATA_ROOT, "--model", "microsoft/deberta-v3-base", "--out-dir", RUNS_DIR]
    process = subprocess.Popen(cmd, cwd="/kaggle/working/repo", env=env,
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    return process, log_path


# what is left to run
todo = []
for seed in SEEDS:
    run_dir = f"{RUNS_DIR}/{RUN_NAME}_seed{seed}"
    if os.path.exists(f"{run_dir}/metrics.json"):
        print("finished already, skipped:", os.path.basename(run_dir))
        continue
    if os.path.exists(run_dir):
        print("interrupted run, deleted and started again:", os.path.basename(run_dir))
        shutil.rmtree(run_dir)
    todo.append(seed)

# running[gpu] = (process , log path) , or None when that GPU is free
running = {0: None, 1: None}
while len(todo) > 0 or any(job is not None for job in running.values()):
    for gpu in running:
        job = running[gpu]
        if job is not None and job[0].poll() is not None:
            process, log_path = job
            print(f"{os.path.basename(log_path)} finished with exit code {process.returncode}", flush=True)
            if process.returncode != 0:
                print(open(log_path).read()[-3000:], flush=True)  # the end of the log , to see the error
            running[gpu] = None
        if running[gpu] is None and len(todo) > 0:
            seed = todo.pop(0)
            running[gpu] = start(gpu, seed)
            print(f"GPU {gpu}: started {RUN_NAME}_seed{seed}", flush=True)
    time.sleep(120)
    for gpu, job in running.items():
        if job is not None:
            lines = open(job[1]).read().strip().splitlines()
            print(f"{os.path.basename(job[1])}: {lines[-1] if lines else '(no output yet)'}", flush=True)
print("all runs finished")

## 7. Results
Per seed: val macro-F1 (6) at k = 3 next to the same seed at k = 0 (exp03), then the mean ± std over the seeds. Then every run's per-class F1, val curve, time per epoch and truncated training inputs.

Reference: k = 0 (exp03, lr 1e-5, 3 epochs) = 0.5225 ± 0.0059.

In [ ]:
import json, statistics, re

scores = []
for seed in SEEDS:
    run_dir = f"{RUNS_DIR}/{RUN_NAME}_seed{seed}"
    if not os.path.exists(f"{run_dir}/metrics.json"):
        print(f"seed {seed}: missing")
        continue
    m = json.load(open(f"{run_dir}/metrics.json"))
    score = m["primary"]["macro_f1_6"]
    scores.append(score)
    print(f"seed {seed}: k = {PAST_K} {score:.4f} (epoch {m['selected_epoch']}) | k = 0 {K0_REFERENCE[seed]:.4f} "
          f"| difference {score - K0_REFERENCE[seed]:+.4f}")
if len(scores) > 1:
    print(f"mean {statistics.mean(scores):.4f} ± {statistics.stdev(scores):.4f} over {len(scores)} seeds "
          f"(k = 0 : 0.5225 ± 0.0059)")

for seed in SEEDS:
    run_dir = f"{RUNS_DIR}/{RUN_NAME}_seed{seed}"
    if not os.path.exists(f"{run_dir}/metrics.json"):
        continue
    m = json.load(open(f"{run_dir}/metrics.json"))
    print(f"\n=== {os.path.basename(run_dir)} | selected epoch {m['selected_epoch']} "
          f"| macro_f1_6 {m['primary']['macro_f1_6']:.4f} | macro_f1_7 {m['macro_f1_7']:.4f}")
    print("per-class F1:", {c: round(v["f1"], 3) for c, v in m["per_class"].items()})
    curve = []
    minutes = []
    for line in open(f"{run_dir}/flat/history.jsonl"):
        h = json.loads(line)
        if h["type"] == "epoch":
            curve.append(f"{h['val_macro_f1']:.3f}")
            minutes.append(f"{h['seconds'] / 60:.1f}")
    print("val macro F1 per epoch:", " , ".join(curve), "| minutes per epoch:", " , ".join(minutes))
    for log_path in glob.glob(f"{LOGS_DIR}/{RUN_NAME}_seed{seed}_gpu*.log"):
        for line in open(log_path):
            if "truncated train inputs" in line:
                print(line.strip())

## 8. Download the run folders
One zip with every exp04 run folder (settings, history, predictions, metrics) and the logs, in the notebook's output files.

In [ ]:
!cd /kaggle/working && zip -qr exp04_runs.zip runs/exp04_* logs && ls -lh exp04_runs.zip